# Crosspost re-collection (pilot, 998 accounts)

The stored history pull did not keep Arctic Shift's crosspost fields. This notebook re-collects them for every post of the
pilot sample (`sample_998_v4_excl_output/sample_posts.parquet`) with `posts/ids` (100 ids per request) and adds manual
cross-posts, for variable #3 (crossposting) and to exclude crossposts from #14 (repeated content).

- **native crosspost**: Arctic Shift `crosspost_parent` is set (Reddit's crosspost feature; present from 2017). The parent's
  author, subreddit, creation time and title come from `crosspost_parent_list[0]`, so own-post vs other-post crossposts can
  be told apart.
- **manual x-post** (any year, including before 2017): the post's title or body says "x-post", "xpost", "x post",
  "cross-post", "crosspost" or "cross post" (e.g. "[x-post from r/pics]").
- `is_crosspost` = either.

Progress is checkpointed to `crosspost_output/checkpoint_native.parquet` so a rerun resumes. Network calls retry up to 10
times and go through a rate-limited session. Final output (never overwritten): `crosspost_output/crosspost_flags_998.parquet`.

In [ ]:
# Libraries.
import pandas as pd
import numpy as np
import os
import re
import time
from requests_ratelimiter import LimiterSession
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

# Folders, checkpoint file, the Arctic Shift posts-by-id endpoint, and ids per request.
SAMPLE_DIR = 'sample_998_v4_excl_output'
OUT_DIR = 'crosspost_output'
CHECKPOINT = os.path.join(OUT_DIR, 'checkpoint_native.parquet')
API = 'https://arctic-shift.photon-reddit.com/api/posts/ids'
BATCH = 100
# Text that marks a manual crosspost: x-post, xpost, crosspost, cross-posted, and so on.
XPOST = re.compile(r'(?<![a-z])(x-?\s?post(?:ed)?|cross-?\s?post(?:ed)?)(?![a-z])', re.I)
os.makedirs(OUT_DIR, exist_ok=True)
# At most 2 requests per second.
session = LimiterSession(per_second=2)

# Write a parquet file only if it does not exist yet (never overwrite).
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

In [ ]:
# Every stored post of the pilot sample (one row per post id).
posts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_posts.parquet'),
                        columns=['id', 'outer_key', 'created_utc', 'subreddit', 'title', 'selftext'])
posts = posts.drop_duplicates('id').reset_index(drop=True)
print(f'{len(posts):,} posts from {posts.outer_key.nunique()} accounts')

In [ ]:
# Fetch up to 100 posts by id; retry up to 10 times with a growing wait.
def fetch(ids):
    retry_count = 0
    while True:
        try:
            r = session.get(API, params={'ids': ','.join(ids)}, timeout=60)
            r.raise_for_status()
            j = r.json()
            if j.get('error'):
                raise RuntimeError(j['error'])
            return j.get('data') or []
        except Exception:
            retry_count += 1
            if retry_count >= 10:
                raise
            time.sleep(5 * retry_count)

# Crosspost fields of one returned post: the parent's fullname, and the parent's author, subreddit, time and title.
def row(d):
    par = (d.get('crosspost_parent_list') or [{}])[0] or {}
    return {'id': d.get('id'), 'returned': True, 'crosspost_parent': d.get('crosspost_parent'),
            'parent_author': par.get('author'), 'parent_subreddit': par.get('subreddit'),
            'parent_created_utc': par.get('created_utc'), 'parent_title': par.get('title')}

# Resume from the checkpoint and fetch only posts not done yet.
done = pd.read_parquet(CHECKPOINT) if os.path.exists(CHECKPOINT) else pd.DataFrame(columns=['id'])
todo = [i for i in posts['id'] if i not in set(done['id'])]
print(f'already done {len(done):,}, to fetch {len(todo):,}')
buffer = []
for k in range(0, len(todo), BATCH):
    ids = todo[k:k + BATCH]
    got = {d.get('id'): row(d) for d in fetch(ids)}
    # Posts Arctic Shift did not return are recorded as returned = False.
    buffer += [got.get(i, {'id': i, 'returned': False}) for i in ids]
    # Write the checkpoint every 20 batches and at the end.
    if len(buffer) >= 20 * BATCH or k + BATCH >= len(todo):
        done = pd.concat([done, pd.DataFrame(buffer)], ignore_index=True)
        done.to_parquet(CHECKPOINT, engine='pyarrow')   # checkpoint: progress file, rewritten as it grows
        buffer = []
        print(f'{len(done):,} / {len(posts):,}', flush=True)

In [ ]:
# Join the fetched fields onto the posts and flag crossposts two ways:
# native (Reddit's crosspost_parent is set) or manual (title or first 500 characters of text say x-post/crosspost).
native = pd.read_parquet(CHECKPOINT).drop_duplicates('id')
flags = posts.merge(native, on='id', how='left', validate='one_to_one')
flags['returned'] = flags['returned'].fillna(False).astype(bool)
flags['native_crosspost'] = flags['crosspost_parent'].notna()
text = flags['title'].fillna('') + ' ' + flags['selftext'].fillna('').str.slice(0, 500)
flags['manual_xpost'] = text.str.contains(XPOST)
flags['xpost_text'] = text.str.extract(r'(.{0,30}(?:x-?\s?post|cross-?\s?post).{0,30})', flags=re.I)[0]
flags['is_crosspost'] = flags['native_crosspost'] | flags['manual_xpost']
# Native crossposts of the account's own earlier post.
flags['crosspost_of_own_post'] = flags['native_crosspost'] & flags['parent_author'].str.lower().eq(flags['outer_key'].str.lower())
# Counts by year.
year = pd.to_datetime(flags['created_utc'], unit='s').dt.year
summary = pd.DataFrame({'posts': flags.groupby(year).size(), 'returned_by_arctic': flags.groupby(year)['returned'].sum(),
                        'native': flags.groupby(year)['native_crosspost'].sum(), 'manual_xpost': flags.groupby(year)['manual_xpost'].sum(),
                        'is_crosspost': flags.groupby(year)['is_crosspost'].sum()})
summary['pct_crosspost'] = (100 * summary['is_crosspost'] / summary['posts']).round(1)
print('native crossposts of the account\'s own post:', int(flags['crosspost_of_own_post'].sum()), 'of', int(flags['native_crosspost'].sum()))
print('accounts with any crosspost before accusation-agnostic count:', flags.loc[flags['is_crosspost'], 'outer_key'].nunique())
summary

In [ ]:
# 15 random manual crossposts, showing the matched text, to check the pattern.
flags[flags['manual_xpost']].sample(min(15, int(flags['manual_xpost'].sum())), random_state=2026)[['subreddit', 'xpost_text']]

In [ ]:
# Save the flags (post title and text dropped).
save_new(flags.drop(columns=['title', 'selftext']), 'crosspost_flags_998.parquet')